In [ ]:
"""
Hybrid CNN-ViT Grid Search

This script performs grid search over hyperparameters (learning rate and batch size) 
for the Hybrid CNN-ViT model on wheat disease detection.
"""

import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import shutil
from itertools import product
from pathlib import Path
from typing import Dict, List
try:
    import timm
except ImportError:
    import sys
    os.system(f"{sys.executable} -m pip install timm")
    import timm
from timm import create_model

# -----------------------------
# Configuration
# -----------------------------
# Auto-detect project root
# Try to find project root by looking for common markers
current_dir = os.getcwd()
if 'train_scripts' in current_dir:
    # We're in train_scripts, go up 2 levels
    PROJECT_ROOT = os.path.abspath(os.path.join(current_dir, '../..'))
elif 'epoch20' in current_dir:
    # We're in epoch20, go up 1 level
    PROJECT_ROOT = os.path.abspath(os.path.join(current_dir, '..'))
else:
    # Try to find from __file__ if running as script
    try:
        SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
        PROJECT_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, '../..'))
    except:
        PROJECT_ROOT = os.path.abspath(os.path.join(current_dir, '../..'))

# Paths relative to project root
DATASET_DIR = os.path.join(PROJECT_ROOT, 'dataset')
SAVE_DIR = os.path.join(PROJECT_ROOT, 'saved_models_and_data')
SPLIT_OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'dataset_split')

# Convert to absolute paths
DATASET_DIR = os.path.abspath(DATASET_DIR)
SAVE_DIR = os.path.abspath(SAVE_DIR)
SPLIT_OUTPUT_DIR = os.path.abspath(SPLIT_OUTPUT_DIR)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32  # Default, will be overridden in grid search
EPOCHS = 20
LEARNING_RATE = 1e-4  # Default, will be overridden in grid search
EARLY_STOPPING_PATIENCE = 5
USE_MIXED_PRECISION = True if torch.cuda.is_available() else False

# Grid search configuration
GRID_SEARCH_DIR = os.path.join(SAVE_DIR, 'grid_search_hybrid_cnn_vit')

# Create directories
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(SPLIT_OUTPUT_DIR, exist_ok=True)
os.makedirs(GRID_SEARCH_DIR, exist_ok=True)

# Print paths for debugging
print(f"Dataset directory: {DATASET_DIR}")
print(f"Save directory: {SAVE_DIR}")
print(f"Split directory: {SPLIT_OUTPUT_DIR}")
print(f"Dataset exists: {os.path.exists(DATASET_DIR)}")

# -----------------------------
# Transformations
# -----------------------------
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
test_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# -----------------------------
# Custom Dataset
# -----------------------------
class WheatDiseaseDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        # Only include subdirectories as classes
        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        self.samples = []
        for target_class in self.classes:
            class_dir = os.path.join(root_dir, target_class)
            for img_file in os.listdir(class_dir):
                path = os.path.join(class_dir, img_file)
                self.samples.append((path, self.class_to_idx[target_class]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        try:
            image = Image.open(path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            return image, target
        except Exception as e:
            print(f"Erreur lors du chargement de {path}: {e}")
            return self.__getitem__((idx + 1) % len(self))

# -----------------------------
# Hybrid Model Definition
# -----------------------------
class HybridCNNViT(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # CNN branch (ConvNeXt)
        self.cnn = models.convnext_base(pretrained=True)
        cnn_out = self.cnn.classifier[2].in_features
        self.cnn.classifier = nn.Identity()
        # ViT branch (from timm)
        self.vit = create_model('vit_base_patch16_224', pretrained=True)
        vit_out = self.vit.head.in_features
        self.vit.head = nn.Identity()
        # Attention-based fusion
        self.fusion = nn.Sequential(
            nn.Linear(cnn_out + vit_out, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        cnn_feat = self.cnn.features(x)  # (batch, C, H, W)
        cnn_feat = cnn_feat.mean(dim=[2, 3])  # Global average pool to (batch, C)
        vit_feat = self.vit(x)  # (batch, features)
        fused = torch.cat([cnn_feat, vit_feat], dim=1)
        out = self.fusion(fused)
        return out

# -----------------------------
# Data Loaders
# -----------------------------
def get_data_loaders(batch_size: int = BATCH_SIZE):
    split_dirs = [os.path.join(SPLIT_OUTPUT_DIR, split) for split in ['train', 'val', 'test']]
    split_exists = all(os.path.isdir(d) and len(os.listdir(d)) > 0 for d in split_dirs)
    if split_exists:
        print('Found existing split dataset. Loading splits...')
        train_dataset = WheatDiseaseDataset(os.path.join(SPLIT_OUTPUT_DIR, 'train'), transform=train_transform)
        val_dataset = WheatDiseaseDataset(os.path.join(SPLIT_OUTPUT_DIR, 'val'), transform=test_transform)
        test_dataset = WheatDiseaseDataset(os.path.join(SPLIT_OUTPUT_DIR, 'test'), transform=test_transform)
    else:
        print('No split dataset found. Splitting and saving images...')
        full_dataset = WheatDiseaseDataset(DATASET_DIR, transform=train_transform)
        generator = torch.Generator().manual_seed(42)
        indices = torch.randperm(len(full_dataset), generator=generator).tolist()
        train_size = int(0.7 * len(full_dataset))
        val_size = int(0.15 * len(full_dataset))
        test_size = len(full_dataset) - train_size - val_size
        train_indices = indices[:train_size]
        val_indices = indices[train_size:train_size + val_size]
        test_indices = indices[train_size + val_size:]
        train_data = Subset(full_dataset, train_indices)
        val_data = Subset(full_dataset, val_indices)
        test_data = Subset(full_dataset, test_indices)
        
        def save_split_images(dataset, indices, split_name):
            print(f"Saving images for split: {split_name}")
            for idx in indices:
                path, label_idx = dataset.dataset.samples[idx]  # dataset is a Subset
                class_name = dataset.dataset.classes[label_idx]
                filename = os.path.basename(path)
                dest_dir = os.path.join(SPLIT_OUTPUT_DIR, split_name, class_name)
                os.makedirs(dest_dir, exist_ok=True)
                dest_path = os.path.join(dest_dir, filename)
                shutil.copyfile(path, dest_path)
        
        save_split_images(train_data, train_indices, 'train')
        save_split_images(val_data, val_indices, 'val')
        save_split_images(test_data, test_indices, 'test')
        print('Image splits saved.')
        train_dataset = WheatDiseaseDataset(os.path.join(SPLIT_OUTPUT_DIR, 'train'), transform=train_transform)
        val_dataset = WheatDiseaseDataset(os.path.join(SPLIT_OUTPUT_DIR, 'val'), transform=test_transform)
        test_dataset = WheatDiseaseDataset(os.path.join(SPLIT_OUTPUT_DIR, 'test'), transform=test_transform)
    
    print('Calculating class weights for balanced sampling...')
    targets = [s[1] for s in train_dataset.samples]
    class_counts = np.bincount(targets)
    class_weights = 1. / class_counts
    sample_weights = [class_weights[t] for t in targets]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    print('Data loaders are ready.')
    return train_loader, val_loader, test_loader, train_dataset.classes

# -----------------------------
# Training Loop
# -----------------------------
def train_model(model, device, train_loader, val_loader, num_epochs=EPOCHS, 
                learning_rate=LEARNING_RATE, trial_name=None):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)
    best_acc = 0.0
    no_improvement_epochs = 0
    scaler = torch.cuda.amp.GradScaler(enabled=USE_MIXED_PRECISION)
    train_log = []
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        running_corrects = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            autocast_device = 'cuda' if torch.cuda.is_available() else 'cpu'
            with torch.amp.autocast(autocast_device, enabled=USE_MIXED_PRECISION):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_corrects.float() / len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                with torch.amp.autocast(autocast_device, enabled=USE_MIXED_PRECISION):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)
                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)
        val_loss = val_loss / len(val_loader.dataset)
        val_acc = val_corrects.float() / len(val_loader.dataset)
        scheduler.step(val_loss)
        
        # Log epoch results
        epoch_log = {
            'epoch': epoch + 1,
            'train_loss': float(epoch_loss),
            'train_acc': float(epoch_acc),
            'val_loss': float(val_loss),
            'val_acc': float(val_acc)
        }
        train_log.append(epoch_log)
        
        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
        
        # Early Stopping
        if val_acc > best_acc:
            best_acc = val_acc
            if trial_name:
                model_path = os.path.join(SAVE_DIR, f"best_hybrid_model_{trial_name.replace(' ', '_')}.pth")
            else:
                model_path = os.path.join(SAVE_DIR, "best_hybrid_model.pth")
            torch.save(model.state_dict(), model_path)
            no_improvement_epochs = 0
        else:
            no_improvement_epochs += 1
        if no_improvement_epochs >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break
    print("Training complete.")
    return model, train_log, best_acc

# -----------------------------
# Evaluation Function
# -----------------------------
def evaluate_on_test(model, device, test_loader, class_labels):
    """Evaluate model on test set and return confusion matrix and classification report"""
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    conf_matrix = confusion_matrix(y_true, y_pred)
    report = classification_report(
        y_true, y_pred, target_names=class_labels, digits=4, output_dict=True
    )
    return conf_matrix, report

# -----------------------------
# Grid Search Function
# -----------------------------
def run_grid_search(
    learning_rates: List[float],
    batch_sizes: List[int],
    num_epochs: int = EPOCHS,
):
    """Run grid search over learning rates and batch sizes"""
    results: List[Dict] = []
    trial_id = 0

    for lr, batch_size in product(learning_rates, batch_sizes):
        trial_id += 1
        trial_name = f"trial{trial_id}_lr{lr}_bs{batch_size}"
        print("\n" + "=" * 80)
        print(f"Starting {trial_name}")
        print("=" * 80)

        train_loader, val_loader, test_loader, class_labels = get_data_loaders(
            batch_size=batch_size
        )
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = HybridCNNViT(num_classes=len(class_labels)).to(device)

        model, train_log, best_val_acc = train_model(
            model=model,
            device=device,
            train_loader=train_loader,
            val_loader=val_loader,
            num_epochs=num_epochs,
            learning_rate=lr,
            trial_name=trial_name,
        )

        # Load best weights for this trial
        best_model_path = os.path.join(
            SAVE_DIR, f"best_hybrid_model_{trial_name.replace(' ', '_')}.pth"
        )
        if os.path.exists(best_model_path):
            model.load_state_dict(
                torch.load(best_model_path, map_location=device)
            )

        conf_matrix, report = evaluate_on_test(
            model, device, test_loader, class_labels
        )

        # Compute overall test accuracy from report
        test_acc = report["accuracy"]

        trial_result = {
            "trial_name": trial_name,
            "learning_rate": lr,
            "batch_size": batch_size,
            "best_val_acc": float(best_val_acc),
            "test_acc": float(test_acc),
        }
        results.append(trial_result)

        # Save per-trial logs
        trial_dir = Path(GRID_SEARCH_DIR) / trial_name
        trial_dir.mkdir(parents=True, exist_ok=True)
        with open(trial_dir / "train_log.json", "w") as f:
            json.dump(train_log, f, indent=2)
        with open(trial_dir / "classification_report.json", "w") as f:
            json.dump(report, f, indent=2)
        np.save(trial_dir / "confusion_matrix.npy", conf_matrix)

    # Sort results by validation accuracy
    results_sorted = sorted(results, key=lambda x: x["best_val_acc"], reverse=True)
    leaderboard_path = Path(GRID_SEARCH_DIR) / "leaderboard.json"
    leaderboard_path.parent.mkdir(parents=True, exist_ok=True)
    with open(leaderboard_path, "w") as f:
        json.dump(results_sorted, f, indent=2)

    print("\n" + "=" * 80)
    print("Grid search completed. Top configurations:")
    print("=" * 80)
    for i, r in enumerate(results_sorted[:5], 1):
        print(
            f"{i}. {r['trial_name']} | lr={r['learning_rate']} | "
            f"bs={r['batch_size']} | best_val_acc={r['best_val_acc']:.4f} | "
            f"test_acc={r['test_acc']:.4f}"
        )
    
    return results_sorted

# -----------------------------
# Main - Grid Search
# -----------------------------
if __name__ == '__main__':
    print("="*80)
    print("HYBRID CNN-ViT GRID SEARCH - WHEAT DISEASE DETECTION")
    print("="*80)
    
    # Grid search configuration
    lr_list = [1e-4, 5e-5]
    batch_size_list = [16, 32]
    
    print(f"\nGrid Search Configuration:")
    print(f"  Learning Rates: {lr_list}")
    print(f"  Batch Sizes: {batch_size_list}")
    print(f"  Total Trials: {len(lr_list) * len(batch_size_list)}")
    print("="*80 + "\n")
    
    # Run grid search
    results = run_grid_search(learning_rates=lr_list, batch_sizes=batch_size_list, num_epochs=EPOCHS)
    
    print("\n" + "="*80)
    print("✓ GRID SEARCH COMPLETE!")
    print("="*80)
    
    # Optional: Evaluate best model and generate confusion matrix
    if results:
        best_trial = results[0]
        print(f"\nBest configuration: {best_trial['trial_name']}")
        print(f"  Validation Accuracy: {best_trial['best_val_acc']:.4f}")
        print(f"  Test Accuracy: {best_trial['test_acc']:.4f}")
        
        # Load best model for visualization
        best_model_path = os.path.join(
            SAVE_DIR, f"best_hybrid_model_{best_trial['trial_name'].replace(' ', '_')}.pth"
        )
        if os.path.exists(best_model_path):
            print(f"\nLoading best model from: {best_model_path}")
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            train_loader, val_loader, test_loader, class_labels = get_data_loaders(
                batch_size=best_trial['batch_size']
            )
            model = HybridCNNViT(num_classes=len(class_labels)).to(device)
            model.load_state_dict(torch.load(best_model_path, map_location=device))
            
            # Generate confusion matrix
            conf_matrix, report = evaluate_on_test(model, device, test_loader, class_labels)
            
            import matplotlib.pyplot as plt
            import seaborn as sns
            plt.figure(figsize=(12, 10))
            sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", 
                        xticklabels=class_labels, yticklabels=class_labels)
            plt.xlabel("Predicted")
            plt.ylabel("Actual")
            plt.title(f"Confusion Matrix - {best_trial['trial_name']}")
            plt.tight_layout()
            confusion_matrix_path = os.path.join(GRID_SEARCH_DIR, f"{best_trial['trial_name']}_confusion_matrix.png")
            plt.savefig(confusion_matrix_path)
            print(f"Confusion matrix saved to: {confusion_matrix_path}")
            plt.show()
            
            print("\nClassification Report:")
            # Re-evaluate to get y_true and y_pred for printing
            model.eval()
            y_true, y_pred = [], []
            with torch.no_grad():
                for inputs, labels in test_loader:
                    inputs = inputs.to(device)
                    labels = labels.to(device)
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    y_true.extend(labels.cpu().numpy())
                    y_pred.extend(preds.cpu().numpy())
            
            print(classification_report(y_true, y_pred, target_names=class_labels, digits=4))



Microsoft Visual C++ Redistributable is not installed, this may lead to the DLL load failure.
It can be downloaded at https://aka.ms/vs/17/release/vc_redist.x64.exe


OSError: [WinError 126] The specified module could not be found. Error loading "C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.